# Session 1: When Optimal Fails, Stress-Testing Classical Minimum-Variance Portfolios

Modern portfolio theory promises an "optimal" allocation, but optimal under what assumptions? In this session, we'll build a classical minimum-variance portfolio and then systematically break it. We'll use AI-driven hybrid Monte Carlo simulation to expose how fragile textbook solutions can be when the world doesn't cooperate.

> __Learning Objectives:__
>
> * __From Prices to Portfolios:__ Derive log growth rates from price data, then estimate portfolio reward and risk using the expected growth rate vector and covariance matrix. Use the Single Index Model (SIM) to construct a factor-based covariance matrix with bootstrap uncertainty quantification.
> * __Optimization and the Sharpe Ratio:__ Solve the Markowitz minimum-variance quadratic program and trace the efficient frontier. Find the tangency (maximum Sharpe ratio) portfolio via second-order cone programming.
> * __Forward Stress Testing and Tail-Risk Scorecard:__ Generate a Monte Carlo ensemble of synthetic futures via the hybrid SIM construction and run buy-and-hold across each path. Score the resulting wealth distribution with formal tail-risk metrics (VaR, CVaR, max drawdown) and the portfolio Net Present Value relative to a risk-free baseline.

Let's get started!
___

## Examples
The following example notebooks accompany this lecture. They contain executable code that implements the concepts discussed here:

[▶ Let's explore the stylized facts of log growth rate data](eCornell-AI-Finance-S1-Example-StylizedFacts-Example-May-2026.ipynb). We compute growth rates from the synthetic training dataset, test for fat tails via Anderson-Darling and Hill's estimator, and verify volatility clustering via autocorrelation of absolute returns.

[▶ Let's build a classical minimum-variance portfolio from synthetic data](eCornell-AI-Finance-S1-Example-BuildMinVariancePortfolio-May-2026.ipynb). We generate a synthetic asset universe, estimate $\mathbb{E}[\mathbf{g}]$ and $\boldsymbol{\Sigma}$, solve the quadratic program, compute the efficient frontier, and explore input sensitivity.

[▶ Let's stress-test the minimum-variance portfolio](eCornell-AI-Finance-S1-Example-StressTestMinVariancePortfolio-May-2026.ipynb). We generate 5,000 synthetic futures via the hybrid SIM construction, run buy-and-hold across four portfolios (min-var, equal-weight 1/N, the market index, and a continuously compounded risk-free baseline), and produce a tail-risk scorecard featuring VaR, CVaR, max drawdown, and portfolio NPV against the risk-free baseline.

[▶ Let's estimate SIM parameters with bootstrap uncertainty quantification](eCornell-AI-Finance-S1-Example-SIMParameterEstimation-May-2026.ipynb). We estimate Single Index Model parameters from the synthetic training dataset via regularized OLS, bootstrap the sampling distribution for a demonstration ticker, and then run batch estimation across all 424 tickers and save the results.

___

## From Prices to Growth Rates
The inputs to portfolio optimization, expected growth rates and covariances, must be _estimated_ from observed price data, starting with the __Continuously Compounded Growth Rate (CCGR)__, which converts a time series of asset prices into a stationary series of growth rates suitable for statistical analysis:

>  __Continuously Compounded Growth Rate (CCGR)__
>
> Let's assume a model of the share price of firm $i$ is governed by an expression of the form:
>$$
\begin{align*}
S^{(i)}_{j} &= S^{(i)}_{j-1}\;\exp\left(\underbrace{g^{(i)}_{j,j-1}\Delta{t}_{j}}_{\text{return}}\right)
\end{align*}
> $$
> where $S^{(i)}_{j-1}$ denotes the share price of firm $i$ at time index $j-1$, $S^{(i)}_{j}$ denotes the share price of firm $i$ at time index $j$, and $\Delta{t}_{j} = t_{j} - t_{j-1}$ denotes the length of a time step (units: years) between time index $j-1$ and $j$. 

The value we are going to estimate from data is the growth rate $g^{(i)}_{j,j-1}$ (units: inverse years) for each firm $i$. Let's rearrange the continuous share price expression to solve for $g^{(i)}_{j,j-1}$:
$$\begin{align*}
S^{(i)}_{j} & = S^{(i)}_{j-1}\;\exp\left(g^{(i)}_{j,j-1}\Delta{t}_{j}\right) \\
S^{(i)}_{j}/S^{(i)}_{j-1} & = \exp\left(g^{(i)}_{j,j-1}\Delta{t}_{j}\right) \\
\ln\left(S^{(i)}_{j}/S^{(i)}_{j-1}\right) & = g^{(i)}_{j,j-1}\Delta{t}_{j} \\
g^{(i)}_{j,j-1} & = \left(\frac{1}{\Delta{t}_{j}}\right)\,\ln\left(\frac{S^{(i)}_{j}}{S^{(i)}_{j-1}}\right)\quad\blacksquare
\end{align*}
$$

Log growth rates are additive over time and approximate simple percentage returns for small values. Given $T$ observations for each of $N$ assets, we assemble the growth rates into a _data matrix_ $\mathbf{X} \in \mathbb{R}^{(T-1) \times N}$ where each row is a time step and each column is an asset. This matrix is the raw material for estimating the inputs to the optimizer.

> __Example__
> 
> Growth rate distributions have some interesting statistical properties (stylized facts) worth exploring:
>
> * __Heavy (fat) tailed distribution__: Extreme price movements are more likely than a Normal distribution predicts.
> * __Absence of Autocorrelation__: Returns are approximately uncorrelated, consistent with a random walk with occasional jumps.
> * __Volatility clustering__: Large price movements tend to be followed by other large moves, and small moves follow small moves.
>
> [▶ Let's explore the stylized facts of log growth rate data](eCornell-AI-Finance-S1-Example-StylizedFacts-Example-May-2026.ipynb). We compute growth rates from the synthetic training dataset, test for fat tails via Anderson-Darling and Hill's estimator, and verify volatility clustering via autocorrelation of absolute returns.
>
> For a deeper dive into the stylized facts of growth rate data: [check out the stylized facts deeper dive notebook](eCornell-AI-Finance-S1-DeeperDive-StylizedFacts-May-2026.ipynb). 

___

<div>
    <center>
        <img src="figs/Fig-MinVar-Portfolio-RA-Schematic.png" width="680"/>
    </center>
</div>

## Modern Portfolio Theory (MPT)
Modern Portfolio Theory (MPT) is a framework for constructing investment portfolios that aims to maximize expected return for a given level of risk or, equivalently, minimize risk for a given level of expected return.

> __Reference:__ Modern Portfolio Theory was introduced by Harry Markowitz in the 1950s and has since become a foundational concept in finance. Markowitz was awarded the Nobel Prize in Economic Sciences in 1990 for his pioneering work in this area. The original publication is given here: [Portfolio Selection, The Journal of Finance, Vol. 7, No. 1 (Mar., 1952), pp. 77-91](https://www.jstor.org/stable/2975974). The Nobel Prize information is available here: [The Sveriges Riksbank Prize in Economic Sciences in Memory of Alfred Nobel 1990](https://www.nobelprize.org/prizes/economic-sciences/1990/markowitz/facts/).

The key idea of MPT is to __optimally balance risk and reward__ by diversifying investments across different assets. Let's dig into this argument by first exploring what we mean by risk and reward of a portfolio, starting with reward.

### Portfolio Reward
The reward of a portfolio is measured by its expected growth (return), which is the weighted sum of the expected growth rates (returns) of the individual assets in the portfolio weighted by the fraction of the total portfolio value invested in each asset. Suppose we have a portfolio $\mathcal{P}$ consisting of $M$ assets. Let $w_i\in\mathbb{R}_{\geq 0}$ be the weight of asset $i$ in the portfolio (i.e., the dollar fraction of the total portfolio value invested in asset $i$), and let $\mathbb{E}[g_i]$ be the expected growth rate (scaled return) of asset $i$. Then, the expected growth rate (return) of the portfolio, denoted as $\mathbb{E}[g_{\mathcal{P}}]$, is given by:
$$
\mathbb{E}[g_{\mathcal{P}}] = \sum_{i=1}^{M} w_i \mathbb{E}[g_i]\quad\Longrightarrow\;\mathbf{w}^{\top}\mathbb{E}[\mathbf{g}]
$$
where $M$ is the total number of assets in the portfolio, i.e., $|\mathcal{P}| = M$, the weight vector is $\mathbf{w}^{\top} = [w_1, w_2, \dots, w_M]$, the sum of weights is one, and $\mathbb{E}[\mathbf{g}] = [\mathbb{E}[g_1], \mathbb{E}[g_2], \dots, \mathbb{E}[g_M]]^{\top}$ is the vector of expected growth rates (returns) of the individual assets.

> __Growth Rate versus Return?__  
>
> We use a somewhat strange convention of measuring reward in terms of the expected growth rate $\mathbb{E}[g_i]$ of asset $i$ instead of the more typical expected return $\mathbb{E}[r_i]$. However, regardless of this choice, the reward argument remains the same (it's just scaled by the inverse time step between the two perspectives). 
> 
> Let $g_{\star} = r_{\star}/\Delta{t}$. Then, the expected return of the portfolio is given by:
> $$
\begin{align*}
\mathbb{E}[g_{\mathcal{P}}] & = \sum_{i=1}^{M} w_i\,\mathbb{E}[g_i]\\
& = \sum_{i=1}^{M} w_i\,\mathbb{E}\Bigl[\frac{r_i}{\Delta{t}}\Bigr]\quad\Longrightarrow\text{pull out } \left(\frac{1}{\Delta{t}}\right)\text{ from the sum} \\
& = \left(\frac{1}{\Delta{t}}\right)\,\sum_{i=1}^{M} w_i\,\mathbb{E}[r_i]\\
& = \left(\frac{1}{\Delta{t}}\right)\mathbf{w}^{\top}\mathbb{E}[\mathbf{r}]\quad\blacksquare    
\end{align*}$$
> The expected return $\mathbf{w}^{\top}\mathbb{E}[\mathbf{r}]$ is __dimensionless__, while the expected growth rate of the portfolio has units of $[\text{time}]^{-1}$. We can convert between the two by multiplying or dividing by the time step $\Delta{t}$.

### Portfolio Risk
The risk of a portfolio is (typically) measured by its variance (or standard deviation) of growth rates (returns), which takes into account the variances of the individual assets as well as the covariances between them. The portfolio variance written in terms of growth rates is given by:
$$
\text{Var}(g_{\mathcal{P}}) = \sum_{i=1}^{M} \sum_{j=1}^{M} w_i w_j \text{Cov}(g_i, g_j)\quad\Longrightarrow\;\mathbf{w}^{\top}\mathbf{\Sigma}_{g}\mathbf{w}
$$
where $\text{Cov}(g_i, g_j)$ is the covariance between the growth rates of assets $i$ and $j$, and $\mathbf{\Sigma}_{g}$ is the covariance matrix of asset growth rates, defined as:
$$
\mathbf{\Sigma}_{g} =
\begin{bmatrix}
\text{Var}(g_1) & \text{Cov}(g_1, g_2) & \cdots & \text{Cov}(g_1, g_M) \\
\text{Cov}(g_2, g_1) & \text{Var}(g_2) & \cdots & \text{Cov}(g_2, g_M) \\
\vdots & \vdots & \ddots & \vdots \\
\text{Cov}(g_M, g_1) & \text{Cov}(g_M, g_2) & \cdots & \text{Var}(g_M)
\end{bmatrix}
$$

However, we could also write the portfolio risk with respect to the returns instead of the growth rates, i.e., $\text{Cov}(r_i, r_j)$ and $\text{Var}(r_p)$.

> __Covariance of Growth Rates versus Returns__  
> 
> The portfolio risk argument remains the same. Let $g_{\star} = r_{\star}/\Delta{t}$. Then, the variance of the portfolio of $M$ assets is given by:
> $$
\begin{align*}
\text{Var}(g_{\mathcal{P}}) & = \sum_{i=1}^{M} \sum_{j=1}^{M} w_i w_j \text{Cov}(g_i, g_j)\\
& = \sum_{i=1}^{M} \sum_{j=1}^{M} w_i w_j \text{Cov}\Bigl(\frac{r_i}{\Delta{t}}, \frac{r_j}{\Delta{t}}\Bigr)\quad\Longrightarrow\text{pull out } \frac{1}{(\Delta{t})^2} \\
& = \frac{1}{(\Delta{t})^2} \sum_{i=1}^{M} \sum_{j=1}^{M} w_i w_j \text{Cov}(r_i, r_j)\\
& = \left(\frac{1}{(\Delta{t})^2}\right)\mathbf{w}^{\top}\mathbf{\Sigma_r}\mathbf{w}\quad\blacksquare    
\end{align*}$$
> where $\mathbf{\Sigma_r}$ is the covariance matrix of asset returns. The portfolio variance $\mathbf{w}^{\top}\mathbf{\Sigma_r}\mathbf{w}$ is __dimensionless__, while the portfolio variance of growth rates has units of $[\text{time}]^{-2}$. We can convert between the two by multiplying or dividing by $(\Delta{t})^2$. 

However, there is a catch. We use (by convention) the _annualized_ variance of the portfolio, which means we need to multiply the portfolio variance by the number of time steps in a year, i.e., $N_{\text{steps}} = 1/\Delta{t}$. Thus, the annualized portfolio variance is given by:
$$
\boxed{
\begin{align*}
\text{Var}_{\text{annualized}}(g_{\mathcal{P}}) = \left(\frac{1}{\Delta{t}}\right)\mathbf{w}^{\top}\mathbf{\Sigma_r}\mathbf{w}\quad\blacksquare
\end{align*}
}
$$

Ok, we have our risk and reward measures for a portfolio of $M$ assets. Now, let's see how we can use this information to construct an optimal portfolio that balances these two forces.
___

## Portfolio Reward and Portfolio Risk
A portfolio $\mathcal{P}$ is defined by a weight vector $\mathbf{w} = (w_1, \ldots, w_N)^{\top}$ where $w_i$ is the fraction of total wealth invested in asset $i$. The portfolio's growth rate at time $t_k$ is the weighted sum of the individual asset growth rates:

$$g_{\mathcal{P}}(t_k) = \sum_{i=1}^{N} w_i\, g_i(t_k) = \mathbf{w}^{\top}\mathbf{g}(t_k)$$

### Portfolio Reward
The _expected_ portfolio growth rate is the weighted average of the individual expected growth rates:

$$\boxed{\mathbb{E}\left[g_{\mathcal{P}}\right] = \sum_{i=1}^{N} w_i\,\mathbb{E}\left[g_i\right] = \mathbf{w}^{\top}\mathbb{E}[\mathbf{g}]}$$

where $\mathbb{E}[\mathbf{g}] = \bigl(\mathbb{E}[g_1],\ldots,\mathbb{E}[g_N]\bigr)^{\top}$ is the expected growth rate vector. This is the portfolio's _reward_.

### Portfolio Risk
The _variance_ of the portfolio growth rate measures how much the portfolio's return fluctuates around its expectation:

$$\boxed{\text{Var}\left(g_{\mathcal{P}}\right) = \mathbf{w}^{\top}\boldsymbol{\Sigma}\,\mathbf{w}}$$

where $\boldsymbol{\Sigma} \in \mathbb{R}^{N \times N}$ is the covariance matrix of asset growth rates. The $(i,j)$ entry of $\boldsymbol{\Sigma}$ is $\sigma_{ij} = \text{Cov}(g_i, g_j)$, and the diagonal entry $\sigma_{ii} = \text{Var}(g_i) = \sigma_i^2$.

> __Decomposing $\boldsymbol{\Sigma}$:__
>
> The covariance matrix can be decomposed as $\boldsymbol{\Sigma} = \mathbf{D}\,\mathbf{C}\,\mathbf{D}$ where $\mathbf{D} = \text{diag}(\sigma_1,\ldots,\sigma_N)$ contains the asset volatilities and $\mathbf{C}$ is the _correlation matrix_ with $C_{ij} = \sigma_{ij}/(\sigma_i\sigma_j)$. This decomposition is useful because volatilities and correlations have different estimation properties and behave differently under stress.

## Estimating $\mathbb{E}[\mathbf{g}]$ and $\boldsymbol{\Sigma}$ from Data
In practice, we never know the true expected growth rates or the true covariance structure, so we estimate them from a historical sample. Let $T$ denote the number of observed growth rates for each of $N$ assets (i.e., $T$ rows of the data matrix). Given the data matrix $\mathbf{X} \in \mathbb{R}^{T \times N}$, the standard estimators are:

**Sample mean** (expected growth rate vector):

$$\hat{\mathbf{g}} = \frac{1}{T}\sum_{t=1}^{T}\mathbf{x}_t$$

**Sample covariance matrix**:

$$\boxed{\hat{\boldsymbol{\Sigma}} = \frac{1}{T-1}\sum_{t=1}^{T}\left(\mathbf{x}_t - \hat{\mathbf{g}}\right)\left(\mathbf{x}_t - \hat{\mathbf{g}}\right)^{\top} = \frac{1}{T-1}\tilde{\mathbf{X}}^{\top}\tilde{\mathbf{X}}}$$

where $\tilde{\mathbf{X}}$ is the _centered_ data matrix with the sample mean subtracted from each row.

> __The estimation asymmetry:__
>
> Expected growth rates are hard to estimate because the signal (a few basis points per day) is tiny relative to the noise (daily volatility of 50-150 bps). The covariance matrix is estimated much more precisely because it measures the _spread_ of the data, not its center. This asymmetry is the root cause of the "error maximization" problem: the optimizer's most important input ($\mathbb{E}[\mathbf{g}]$) is its least reliable one.

## The Single Index Model (SIM): A Factor-Based Alternative

The sample covariance matrix $\hat{\boldsymbol{\Sigma}}$ has $N(N+1)/2$ free parameters. For a portfolio of 50 assets, that's 1,275 parameters estimated from a finite sample. The **Single Index Model** (SIM) offers a structured alternative that reduces the number of free parameters from $O(N^2)$ to $O(N)$.

### The SIM Regression

The SIM posits that the growth rate of asset $i$ at time $t$ is:

$$\boxed{g_i(t) = \alpha_i + \beta_i\, g_M(t) + \varepsilon_i(t)}$$

where:
- $\alpha_i$ is **Jensen's alpha**: the firm-specific excess growth rate not explained by the market
- $\beta_i$ is the **market beta**: the asset's sensitivity to the market factor ($\beta > 1$ amplifies market moves, $\beta < 0$ moves opposite)
- $g_M(t)$ is the market index growth rate (e.g., SPY or a synthetic market index)
- $\varepsilon_i(t) \sim \mathcal{N}(0, \Delta t\,\sigma_{\varepsilon,i}^2)$ is idiosyncratic noise, uncorrelated across assets and independent of $g_M$

### The SIM-Derived Covariance Matrix

Since the idiosyncratic errors $\varepsilon_i$ are uncorrelated across assets and independent of $g_M$, the covariance between any two assets flows entirely through the market factor. This yields a structured covariance matrix with only $2N + 1$ free parameters:

$$\boxed{\Sigma_{ij}^{\text{SIM}} = \begin{cases} \beta_i^2\,\sigma_M^2 + \Delta t\,\sigma_{\varepsilon,i}^2 & \text{if } i = j \\ \beta_i\,\beta_j\,\sigma_M^2 & \text{if } i \neq j \end{cases}}$$

> __Why SIM covariance beats sample covariance:__
>
> For $N = 50$ assets, the sample covariance has 1,275 free parameters; SIM has 101. Fewer parameters means lower estimation variance, better conditioning, and more stable portfolio weights. The trade-off is bias: SIM assumes all cross-asset correlation flows through the single market factor, which is an approximation.

For the full details on estimating SIM parameters via regularized OLS, bootstrap uncertainty quantification, and the derivation of the SIM covariance matrix: [check out the SIM deeper dive notebook](eCornell-AI-Finance-S1-DeeperDive-SIM-May-2026.ipynb).

## The Classical Minimum-Variance Portfolio
In 1952, [Harry Markowitz](https://doi.org/10.2307/2975974) proposed a deceptively simple idea: investors care about the _expected return_ and the _variance_ (risk) of their portfolio, and they want the best trade-off between the two. The minimum-variance portfolio is the allocation that achieves the lowest possible risk for a given level of expected return.

With the estimated expected growth rate vector $\hat{\mathbf{g}}$ and covariance matrix $\hat{\boldsymbol{\Sigma}}$ from the previous section in hand, we can now formulate the optimization problem. We seek a weight vector $\mathbf{w}\in\mathbb{R}^{N}$ that solves the quadratic program:

$$\boxed{\begin{align*}\min_{\mathbf{w}} &\quad \mathbf{w}^{\top}\hat{\boldsymbol{\Sigma}}\,\mathbf{w} \\ \text{subject to:} &\quad \mathbf{w}^{\top}\hat{\mathbf{g}} \geq R \\ &\quad \sum_{i=1}^{N} w_{i} = 1 \\ &\quad l_{i} \leq w_{i} \leq u_{i} \quad \forall\, i\end{align*}}$$

where $R$ is the target return and $l_i, u_i$ are the lower and upper bounds on each weight (e.g., no short selling means $l_i = 0$).

> __Interpretation:__
>
> The objective $\mathbf{w}^{\top}\hat{\boldsymbol{\Sigma}}\,\mathbf{w}$ is the portfolio variance. The first constraint ensures we achieve at least a target return $R$. The budget constraint forces the weights to sum to 1 (fully invested). The box constraints enforce position limits.

This is a convex quadratic program, it has a unique global minimum and can be solved efficiently using solvers such as [Ipopt](https://coin-or.github.io/Ipopt/) or [COSMO](https://github.com/oxfordcontrol/COSMO.jl). On the surface, it seems like the perfect recipe for rational investing. But there's a catch: _everything depends on the quality of $\hat{\mathbf{g}}$ and $\hat{\boldsymbol{\Sigma}}$_, and as we discussed above, $\hat{\mathbf{g}}$ is estimated with far more error than $\hat{\boldsymbol{\Sigma}}$.

### The Efficient Frontier
By sweeping the target return $R$ from its minimum to its maximum feasible value and solving the quadratic program at each point, we trace out the _efficient frontier_, the set of portfolios that achieve the lowest risk for each level of return.



The figure above illustrates the key features of the risk-reward landscape:

- **Portfolio 1** (red, upper frontier) is an efficient portfolio with high expected reward $\mathbb{E}(R_1)$ but also high risk $\sigma_1$. It lies _on_ the efficient frontier.
- **Portfolio 2** (blue, min-var) is the _minimum-variance portfolio_ — it has the lowest achievable risk $\sigma_2$ among all feasible allocations. It anchors the left end of the frontier and earns expected reward $\mathbb{E}(R_2)$.
- **Portfolio 3** (open circle, below frontier) is _suboptimal_: it has the same risk $\sigma_1$ as Portfolio 1, but lower expected reward $\mathbb{E}(R_3)$. A rational investor should move _up_ to the frontier.
- The **solid red curve** is the efficient frontier — every point on it is Pareto optimal. The **dashed curve** below the min-var portfolio represents feasible but dominated allocations.

> __The investor's trade-off:__
>
> Moving along the frontier from Portfolio 2 to Portfolio 1 increases both reward _and_ risk. The curvature of the frontier encodes the _cost of chasing return_: initially, a small increase in risk buys a lot of extra return, but the marginal benefit diminishes as we move further right. The shape of this curve depends entirely on $\hat{\mathbf{g}}$ and $\hat{\boldsymbol{\Sigma}}$, which is why estimation quality matters so much.

### The Sharpe Ratio and the Tangency Portfolio

The efficient frontier tells us the _minimum risk_ for each return level, but which point on the frontier is "best"? One natural criterion is the **Sharpe ratio**, which measures risk-adjusted return:

$$\boxed{\text{SR}_{\mathcal{P}} = \frac{\mathbb{E}[g_{\mathcal{P}}] - g_f}{\sigma_{\mathcal{P}}}}$$

where $g_f$ is the risk-free rate and $\sigma_{\mathcal{P}} = \sqrt{\mathbf{w}^{\top}\boldsymbol{\Sigma}\mathbf{w}}$ is the portfolio volatility. The Sharpe ratio answers: _how much excess return do I earn per unit of risk?_

The portfolio that **maximizes** the Sharpe ratio is called the **tangency portfolio**, the point where a line from the risk-free rate is tangent to the efficient frontier. This line is the **Capital Market Line (CML)**: any point on it can be achieved by mixing the tangency portfolio with the risk-free asset.

> __Why SIM inputs for the Sharpe ratio?__
>
> The excess return vector decomposes naturally into SIM components: $\alpha_i$ (firm-specific skill) + $\beta_i g_M$ (market exposure) $- g_f$ (opportunity cost). Using SIM parameters directly gives a more structured and interpretable decomposition of what drives the tangency portfolio's risk-adjusted return.

For the SOCP formulation used to solve the max-Sharpe problem: [see the SIM deeper dive notebook](eCornell-AI-Finance-S1-DeeperDive-SIM-May-2026.ipynb).

## When Optimal Fails: Three Weaknesses of Mean-Variance Optimization

The minimum-variance framework is elegant, but practitioners have learned, often painfully, that it has three systemic weaknesses:

### 1. Input Sensitivity (Error Maximization)
The optimizer treats $\mathbb{E}[\mathbf{g}]$ and $\boldsymbol{\Sigma}$ as _ground truth_. But these are estimated from historical data, and small estimation errors can produce wildly different optimal weights. [Michaud (1989)](https://doi.org/10.2469/faj.v45.n1.31) coined the term _"error maximizer"_ to describe mean-variance optimization: the optimizer aggressively exploits estimation errors, loading up on assets whose returns are _overestimated_ and avoiding those that are _underestimated_.

> __The Estimation Problem:__
>
> Expected returns are difficult to estimate. A change of just 50 basis points in one asset's expected return can shift its optimal weight by 20% or more. The covariance matrix is more stable, but still subject to estimation error, especially off-diagonal terms (correlations) during periods of market stress. [Chopra and Ziemba (1993)](https://doi.org/10.3905/jpm.1993.409440) showed that errors in means are roughly 10x more damaging than errors in variances.

### 2. Weight Concentration
Without tight constraints, minimum-variance portfolios tend to concentrate in a small number of assets, often the ones with the lowest historical variance. This produces portfolios that _look_ optimal on paper but carry hidden concentration risk.

> __Why does this happen?__
>
> The quadratic objective $\mathbf{w}^{\top}\boldsymbol{\Sigma}\mathbf{w}$ rewards low-variance assets disproportionately. If two assets have similar expected returns but one has slightly lower variance, the optimizer will heavily favor the lower-variance asset. In practice, this means a "diversified" portfolio can end up with 60-80% of its weight in just 2-3 names.

### 3. Assumption Fragility
The entire framework assumes that returns are drawn from a stationary multivariate distribution, that the future will look like the past. This assumption breaks during exactly the moments when portfolio construction matters most: regime changes, market crises, and structural shifts.

> __Stationarity is a luxury:__
>
> Correlations spike during crises (assets that were "diversifying" suddenly move together), volatilities jump, and expected returns shift. The portfolio that was "optimal" under calm conditions can become the _worst_ possible allocation when the regime changes. This phenomenon, sometimes called _correlation breakdown_, is one of the central motivations for the HMM-based approach we'll explore in Session 3.

## Forward Stress Testing via Hybrid Monte Carlo

The classical approach to stress testing perturbs the inputs (correlations, prices, costs) and re-solves the optimization. We take a different approach: we fix the allocation and instead sample many *forward futures* from a calibrated generative model. The optimization itself is no longer the variable; the world is.

### Buy-and-Hold Wealth Dynamics

Given an allocation $\mathbf{w}$ and initial wealth $W_{\mathcal{P}}(0) = B_0$, the share count purchased in asset $i$ at time $t_0$ is fixed by the allocation:

$$\boxed{n_i = \frac{w_i\,W_{\mathcal{P}}(0)}{S_i(t_0)}, \qquad i = 1,\ldots,N}$$

These shares are held unchanged for the entire horizon (no rebalancing — Session 1 is _frictionless and fixed-weight_). The portfolio wealth at any future time $t_k$ is then the share-weighted price aggregate:

$$\boxed{W_{\mathcal{P}}(t_k) = \sum_{i=1}^{N} n_i\,S_i(t_k)}$$

### The Monte Carlo Ensemble

We draw $n_{\text{paths}}$ synthetic price trajectories $\bigl\{S_i^{(p)}(t_k)\bigr\}_{p=1}^{n_{\text{paths}}}$ from the calibrated **hybrid SIM construction**: per-ticker HMM marginals supply the heavy-tailed idiosyncratic dynamics, the SIM regression $g_i = \alpha_i + \beta_i\,g_M + \varepsilon_i$ supplies the market-factor structure, and a Student-$t$ copula reorders the residuals to reproduce realistic cross-sectional dependence. Each synthetic path produces a terminal wealth

$$W_{\mathcal{P}}^{(p)}(T) = \sum_{i=1}^{N} n_i\,S_i^{(p)}(T)$$

and the collection $\bigl\{W_{\mathcal{P}}^{(p)}(T)\bigr\}_{p=1}^{n_{\text{paths}}}$ is the **empirical distribution of outcomes** that the rest of the scorecard works with.

> __Why a distribution, not a number?__
>
> A point estimate of expected return tells us the _center_ of the wealth distribution and nothing about its shape. Tail risk lives in the _tails_ — the worst 5% of outcomes — and you cannot see it without sampling many paths. The hybrid Monte Carlo turns "what is the expected outcome?" into "what does the full distribution of outcomes look like, and how bad is the bad tail?"

___
## Tail-Risk Metrics

The terminal-wealth distribution $\bigl\{W_{\mathcal{P}}^{(p)}(T)\bigr\}$ is summarized by a small set of metrics that focus on the **left tail** — the bad outcomes — rather than the center.

> __Value-at-Risk (VaR) at level $\alpha$:__
>
> The Value-at-Risk at level $\alpha$ is the wealth threshold below which only an $\alpha$ fraction of paths end:
>
> $$\boxed{\text{VaR}_{\alpha}(W_T) = \inf\bigl\{w \in \mathbb{R} : \,P\bigl(W_T \leq w\bigr) \geq \alpha\bigr\}}$$
>
> Equivalently, $\text{VaR}_{\alpha}$ is the $\alpha$-quantile of the terminal-wealth distribution. For $\alpha = 5\%$ it answers: _"what is the worst case I should expect 95% of the time?"_

> __Conditional VaR (CVaR) / Expected Shortfall:__
>
> The Conditional VaR at level $\alpha$ is the **mean** terminal wealth conditional on being in the worst $\alpha$ tail:
>
> $$\boxed{\text{CVaR}_{\alpha}(W_T) = \mathbb{E}\bigl[\,W_T \,\big|\, W_T \leq \text{VaR}_{\alpha}(W_T)\,\bigr]}$$
>
> Where VaR only sees the cutoff, CVaR sees the entire tail. CVaR is also known as **Expected Shortfall (ES)** and is a *coherent* risk measure (it satisfies sub-additivity, which VaR does not), so it is the metric of choice for modern tail-risk reporting.

> __CVaR sample standard error:__
>
> Given $n_{\text{paths}}$ Monte Carlo samples, the tail contains $n_{\text{tail}} = \lfloor \alpha\,n_{\text{paths}}\rfloor$ paths. The plug-in estimator and its analytical standard error are:
>
> $$\boxed{\widehat{\text{CVaR}}_{\alpha} = \frac{1}{n_{\text{tail}}}\sum_{p\,\in\,\text{tail}} W_T^{(p)}, \qquad \text{SE}\bigl(\widehat{\text{CVaR}}_{\alpha}\bigr) \approx \frac{\mathrm{std}(\text{tail})}{\sqrt{n_{\text{tail}}}}}$$
>
> Reporting the standard error alongside the point estimate tells you whether $n_{\text{paths}}$ is large enough to trust the number. A rule of thumb: if the SE-to-CVaR ratio exceeds about 1%, increase $n_{\text{paths}}$ before quoting the figure.

> __Maximum Drawdown:__
>
> Given a path-wise wealth series $W_{\mathcal{P}}^{(p)}(t)$, define the running peak as $\hat{W}^{(p)}(t) = \max_{s \leq t} W_{\mathcal{P}}^{(p)}(s)$. The maximum drawdown along that path is the deepest peak-to-trough fractional decline:
>
> $$\boxed{\text{MaxDD}^{(p)} = \max_{t}\;\frac{\hat{W}^{(p)}(t) - W_{\mathcal{P}}^{(p)}(t)}{\hat{W}^{(p)}(t)}}$$
>
> Drawdown is path-wise, not terminal: a portfolio can end the horizon at break-even after a 30% mid-path drawdown that would have triggered margin calls in real life. The scorecard reports both the median and the 95th percentile of $\text{MaxDD}^{(p)}$ across the $n_{\text{paths}}$ futures.

___
## Portfolio NPV: Comparing to the Risk-Free Baseline

A purely nominal comparison — _"did the portfolio make money?"_ — sets the bar too low. The economically meaningful question is: _did the portfolio beat what the same dollars would have earned sitting in a risk-free asset?_ The right tool for that comparison is the **Net Present Value** of the portfolio.

Following the formulation introduced in the [CHEME-5660 portfolio NPV lecture](https://github.com/varnerlab/CHEME-5660-CourseRepository-Fall-2025/blob/main/lectures/week-6/L6b/CHEME-5660-L6b-Lecture-MAGBM-Data-Portfolios-Fall-2025.ipynb), define the NPV of a portfolio with constant continuous-compounding discount rate $\bar r$ over horizon $T$ as

$$\boxed{\text{NPV}(\bar r, T) = -W_{\mathcal{P}}(0) + W_{\mathcal{P}}(T)\,e^{-\bar r\,T}}$$

The first term is the cash outflow at $t_0$ (the initial investment); the second is the terminal portfolio wealth discounted back to today's dollars. Dividing by the initial wealth gives the **scaled NPV**, the fractional excess in present value per dollar invested:

$$\boxed{\frac{\text{NPV}(\bar r, T)}{W_{\mathcal{P}}(0)} = \frac{W_{\mathcal{P}}(T)}{W_{\mathcal{P}}(0)}\,e^{-\bar r\,T} - 1}$$

For stress-test evaluation we set $\bar r = r_f$, the **risk-free rate**. With this choice the NPV has a sharp economic interpretation:

- $\text{NPV} > 0$ — the risky portfolio **beat** a risk-free STRIPS investment in present-value terms.
- $\text{NPV} < 0$ — the risky portfolio **underperformed** the risk-free baseline; you would have done better holding cash earning $r_f$.
- $\text{NPV} = 0$ — by construction this is the risk-free portfolio itself, $W_{\mathcal{P}}(T) = W_{\mathcal{P}}(0)\,e^{r_f\,T}$. It is the *zero we are measuring against*.

> __NPV-fail rate:__
>
> The economically meaningful failure probability is the fraction of paths on which the portfolio underperformed the risk-free baseline:
>
> $$\boxed{P\bigl[\text{NPV} < 0\bigr] = P\Bigl[W_{\mathcal{P}}(T) < W_{\mathcal{P}}(0)\,e^{r_f\,T}\Bigr]}$$
>
> This strictly dominates the nominal-loss rate $P[W_T < W_{\mathcal{P}}(0)]$ as a measure of failure. A portfolio that grew by 1% nominal over a 5% risk-free year has _failed_ even though it didn't lose money in nominal terms — its capital was idle while the risk-free alternative was earning 4 points more.

Because the hybrid Monte Carlo gives us a distribution $\{W_{\mathcal{P}}^{(p)}(T)\}$, NPV is also a distribution $\{\text{NPV}^{(p)}\}$, and the scorecard reports **median NPV**, the **CVaR of NPV** (worst-tail discounted excess loss), and $P[\text{NPV} < 0]$ alongside the wealth-based metrics. The risk-free portfolio collapses to a single point at NPV $= 0$ and provides the natural reference line on every NPV plot.

## The Baseline Scorecard

Before we can evaluate any improvements (Sessions 2-4), we need a _baseline_ — a quantitative record of how the classical fixed-weight minimum-variance allocation behaves across the hybrid Monte Carlo ensemble. The scorecard tracks **seven** metrics, computed across all $n_{\text{paths}}$ synthetic futures:

| Metric | Definition | What it tells us |
|:-------|:----------|:---------------|
| __Median $W_T$__ | $\text{median}_p\,W_{\mathcal{P}}^{(p)}(T)$ | Center of the terminal-wealth distribution |
| __VaR$_5$__ | 5th-percentile terminal wealth | Worst-case threshold for 95% of futures |
| __CVaR$_5$__ (with SE) | Mean wealth in the worst 5% of paths | Average loss in the bad tail, with sampling SE |
| __Max Drawdown__ (median, P95) | Median and 95th-percentile of $\text{MaxDD}^{(p)}$ across paths | Worst-case path-wise loss along the trajectory |
| __Median Sharpe__ | Median of per-path Sharpe ratios | Risk-adjusted return summary |
| __Median NPV__ | $\text{median}_p\,\text{NPV}^{(p)}(r_f, T)$ | Excess wealth over the risk-free baseline (today's dollars) |
| __NPV-fail rate__ | $P\bigl[\text{NPV} < 0\bigr]$ | Fraction of futures the portfolio underperformed risk-free |

> __Why these seven and not the textbook four?__
>
> Turnover and trading-cost metrics — staples of classical scorecards — are *zero by construction* in Session 1 because the allocation is fixed for the entire horizon (no rebalancing) and we are operating in the frictionless regime. They become meaningful in [Session 2](../session-2/eCornell-AI-Finance-S2-Lecture-AIRebalancingEngine-May-2026.ipynb) when the AI rebalancing engine introduces time-varying weights and transaction costs. In Session 1's frictionless world, the seven metrics above are necessary and sufficient to characterize a fixed-weight allocation's behavior across many futures.

> __Reading the scorecard:__
>
> The risk-free row of the scorecard is intentionally degenerate: deterministic terminal wealth $B_0\,e^{r_f T}$, zero drawdown, zero Sharpe (zero excess vol), $\text{NPV} = 0$, NPV-fail rate $= 0$. It is not a competitor — it is the *zero line* against which every risky portfolio's NPV is measured. A risky portfolio "wins" relative to risk-free if its median NPV is positive *and* its NPV-fail rate is acceptably low (depending on risk tolerance).

The baseline scorecard is computed in the [stress-test example notebook](eCornell-AI-Finance-S1-Example-StressTestMinVariancePortfolio-May-2026.ipynb) and persisted to disk as the bar that Session 2's adaptive rebalancing engine must clear.

## Summary

In this session, we built the full analytical pipeline — from prices to growth rates, through SIM estimation and covariance construction, to minimum-variance and maximum-Sharpe optimization — and then stress-tested the result by running buy-and-hold across thousands of synthetic futures and scoring the wealth distribution against a risk-free baseline.

> __Key Takeaways:__
>
> * __The Single Index Model:__ SIM reduces covariance estimation from quadratic to linear in the number of assets, producing better-conditioned inputs for the optimizer. Bootstrap uncertainty quantification reveals how reliable the alpha and beta estimates actually are.
> * __The tangency portfolio:__ The maximum-Sharpe portfolio sits at the point where the Capital Market Line touches the efficient frontier. It accepts more variance than the min-variance portfolio but delivers substantially better risk-adjusted return.
> * __Forward stress testing and the NPV-aware scorecard:__ The hybrid Monte Carlo turns a single expected-return number into a full distribution of outcomes, and the tail-focused scorecard (VaR, CVaR, max drawdown) plus portfolio NPV reveals where classical mean-variance breaks down. The portfolio NPV relative to the risk-free rate is the economically correct measure of failure — a portfolio that grew in nominal terms but underperformed risk-free has still failed.

___

__What comes next?__ In [Session 2](../session-2/eCornell-AI-Finance-S2-Lecture-AIRebalancingEngine-May-2026.ipynb), we move from static weights to adaptive allocation, designing an AI rebalancing engine that uses Cobb-Douglas utility maximization with sentiment-driven preference weights (built on SIM parameters) to respond dynamically to changing market conditions. The Session 1 scorecard — computed in the stress-test example notebook and saved to `stress-test-baseline.jld2` — is the bar Session 2 must clear.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use synthetic data and simplified models. Real-world portfolio construction involves additional considerations including regulatory requirements, tax implications, liquidity constraints, and operational risks that are beyond the scope of this session.